In [16]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

True

In [17]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Usando o nome curto do modelo para evitar conflitos de prefixo
MODEL_ID = "gemma-4-26b-a4b-it"

# Se o erro 404 persistir, tente usar o gemini-2.0-flash que é mais estável para multimodal
# MODEL_ID = "gemini-2.0-flash"

In [18]:
# Upload do arquivo
file_path = "sales_data.csv"
if not os.path.exists(file_path):
    print(f"Erro: Arquivo {file_path} não encontrado localmente.")
else:
    file = client.files.upload(file=file_path)
    print(f"Arquivo enviado: {file.display_name}")

Arquivo enviado: None


In [19]:
# Aguarda o processamento do arquivo
while file.state.name == "PROCESSING":
    print(".", end="")
    time.sleep(2)
    file = client.files.get(name=file.name)

print(f"\nStatus do arquivo: {file.state.name}")


Status do arquivo: ACTIVE


In [20]:
# Configuração do Chat com Code Execution
chat = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        system_instruction="Você é um analista que analisa dados sobre vendas",
        tools=[types.Tool(code_execution=types.ToolCodeExecution())],
        temperature=0.1
    )
)

In [21]:
pergunta = "Gere um gráfico de pizza com o percentual de vendas por linha de produto"

In [22]:
# Envia a pergunta junto com a referência ao arquivo
try:
    response = chat.send_message([pergunta, file])
    print(response.text)
except Exception as e:
    print(f"Erro ao enviar mensagem: {e}")

Erro ao enviar mensagem: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Requested entity was not found.', 'status': 'NOT_FOUND'}}


In [ ]:
# Analisando os passos e o código executado
if 'response' in locals() and response.candidates:
    for part in response.candidates[0].content.parts:
        if part.executable_code:
            print("\n==== Código Executado ====")
            print(part.executable_code.code)
        if part.code_execution_result:
            print("\n==== Resultado da Execução ====")
            print(part.code_execution_result.output)